## Validation Findings

The data validation phase confirmed that the core structure of the dataset is reliable enough to continue into profitability analysis.

Core structural checks passed successfully, including:

- Missing value checks
- Duplicate record checks
- Negative value checks
- Orphan key checks
- Date and month consistency checks
- Sales-to-cost join coverage checks

Two reconciliation checks failed and were flagged for review:

| Check | Table | Issue Count | Severity | Interpretation |
|---|---:|---:|---|---|
| `cost_components_not_equal_total_cost` | `costs` | 553 | High | Total cost does not match the sum of cost components. |
| `vendor_cost_change_pct_mismatch` | `vendor_prices` | 139 | Medium | Stored cost change percentage does not match recalculated values. |

These issues indicate calculation inconsistencies rather than structural data failures. They were documented and flagged for review before final profitability reporting.

In [ ]:
-- Show results
SELECT * FROM validation_results_final;

In [ ]:
-- Show failures only
SELECT
    test_name,
    table_name,
    issue_count,
    severity,
    status
FROM validation_results_final
WHERE status = 'FAIL';

In [ ]:
-- Show summary of results
SELECT
    status,
    COUNT(*) AS total_checks,
    SUM(issue_count) AS total_issues
FROM validation_results_final
GROUP BY status;

### Decision

The dataset can move forward to analysis, but the two failed reconciliation checks should be reviewed before using `total_cost` and `cost_change_pct` in final executive conclusions.

![Top Issues](../images/validation_top_issues.png)
![Failures](../images/validation_failures.png)
![Summary](../images/validation_summary.png)

## Validation Summary Table (Python)

A cleaned and structured validation summary table was created from SQL outputs using pandas. This table standardizes validation results for reporting and downstream analysis.

In [2]:
## Load CSV file into a DataFrame and display the first few rows
import pandas as pd

df = pd.read_csv('../outputs/validation_results.csv')
df.head()

,test_name,table_name,issue_count,severity,status,action_taken,validated_at
0,cost_components_not_equal_total_cost,costs,553,high,FAIL,Recalculate total_cost from cost components,2026-04-23 19:19:57.495428-04
1,duplicate_costs_rows,costs,0,high,PASS,Deduplicate costs by month-location-product,2026-04-23 19:19:57.495428-04
2,duplicate_sales_rows,sales,0,high,PASS,Deduplicate repeated sales records,2026-04-23 19:19:57.495428-04
3,missing_cogs,costs,0,high,PASS,Investigate null cogs rows before profitabilit...,2026-04-23 19:19:57.495428-04
4,missing_location_id,sales,0,high,PASS,Investigate null location_id and correct sourc...,2026-04-23 19:19:57.495428-04


In [3]:
## Create clean summary table for reporting
validation_summary = df[
    ['test_name','table_name','issue_count','severity','status','action_taken']
].copy()

In [4]:
## Sort properly by severity and issue count for better reporting
severity_order = {'high': 1, 'medium': 2, 'low': 3}

validation_summary['severity_rank'] = validation_summary['severity'].map(severity_order)

validation_summary = validation_summary.sort_values(
    by=['severity_rank', 'issue_count'],
    ascending=[True, False]
).drop(columns='severity_rank')

validation_summary.head(10)

,test_name,table_name,issue_count,severity,status,action_taken
0,cost_components_not_equal_total_cost,costs,553,high,FAIL,Recalculate total_cost from cost components
1,duplicate_costs_rows,costs,0,high,PASS,Deduplicate costs by month-location-product
2,duplicate_sales_rows,sales,0,high,PASS,Deduplicate repeated sales records
3,missing_cogs,costs,0,high,PASS,Investigate null cogs rows before profitabilit...
4,missing_location_id,sales,0,high,PASS,Investigate null location_id and correct sourc...
5,missing_product_id,sales,0,high,PASS,Investigate null product_id and correct source...
6,missing_revenue,sales,0,high,PASS,Investigate null revenue rows before analysis
7,missing_transaction_date,sales,0,high,PASS,Backfill or remove rows with missing transacti...
8,negative_cogs,costs,0,high,PASS,Investigate invalid cost records
9,negative_revenue,sales,0,high,PASS,Investigate refund logic or invalid sales rows


In [5]:
## Export the summary table to a new CSV file for reporting
validation_summary.to_csv('../data/cleaned/validation_summary.csv', index=False)

### Output

The final validation summary dataset was exported to:

`data/cleaned/validation_summary.csv`

This file provides a clean, prioritized view of all validation checks and is used as the foundation for downstream analysis.